In [1]:
import os,json
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        print(f"Checking {p} for project root...")
        if (p / "data").exists():  
            return p
    raise RuntimeError("Project root not found")

BASE_DIR = find_project_root(Path.cwd())
TRUTH_DATASET_PATH = BASE_DIR / "data" / "ground_truth_dataset.json"



Checking d:\Pl_Capitals_Assignment\portfolio-analytics-agent\src\evaluation for project root...
Checking d:\Pl_Capitals_Assignment\portfolio-analytics-agent\src for project root...
Checking d:\Pl_Capitals_Assignment\portfolio-analytics-agent for project root...


In [ ]:
with open(TRUTH_DATASET_PATH, "r") as f:
    truth_dataset = json.load(f)
    
    
    
TRUTH_DATA = truth_dataset
print(truth_dataset)

for key, val in TRUTH_DATA.items():
    print(f'key: {key}, val: {val}')


{'questions': [{'id': 1, 'type': 'text2sql', 'difficulty': 'easy', 'question': 'How many portfolios do we have in total?', 'ground_truth': {'sql_query': 'SELECT COUNT(*) FROM portfolios;', 'expected_result_type': 'single_value', 'explanation': 'Simple COUNT query on portfolios table'}}, {'id': 2, 'type': 'text2sql', 'difficulty': 'easy', 'question': 'What are the names of all active portfolios?', 'ground_truth': {'sql_query': "SELECT portfolio_name FROM portfolios WHERE status = 'Active';", 'expected_result_type': 'list', 'explanation': 'Simple SELECT with WHERE clause filtering by status'}}, {'id': 3, 'type': 'text2sql', 'difficulty': 'easy', 'question': 'Which securities are in the Technology sector?', 'ground_truth': {'sql_query': "SELECT s.symbol, s.company_name FROM securities s JOIN sectors sec ON s.sector_id = sec.sector_id WHERE sec.sector_name = 'Technology';", 'expected_result_type': 'table', 'explanation': 'JOIN between securities and sectors tables with simple WHERE filter'

key: questions, val: [{'id': 1, 'type': 'text2sql', 'difficulty': 'easy', 'question': 'How many portfolios do we have in total?', 'ground_truth': {'sql_query': 'SELECT COUNT(*) FROM portfolios;', 'expected_result_type': 'single_value', 'explanation': 'Simple COUNT query on portfolios table'}}, {'id': 2, 'type': 'text2sql', 'difficulty': 'easy', 'question': 'What are the names of all active portfolios?', 'ground_truth': {'sql_query': "SELECT portfolio_name FROM portfolios WHERE status = 'Active';", 'expected_result_type': 'list', 'explanation': 'Simple SELECT with WHERE clause filtering by status'}}, {'id': 3, 'type': 'text2sql', 'difficulty': 'easy', 'question': 'Which securities are in the Technology sector?', 'ground_truth': {'sql_query': "SELECT s.symbol, s.company_name FROM securities s JOIN sectors sec ON s.sector_id = sec.sector_id WHERE sec.sector_name = 'Technology';", 'expected_result_type': 'table', 'explanation': 'JOIN between securities and sectors tables with simple WHERE 

Select the question Number
1. How many portfolios do we have in total?
2. What are the names of all active portfolios?
3. Which securities are in the Technology sector?
4. What is the total Assets Under Management (AUM) for portfolios with high target risk level?
5. Show me the top 5 holdings by cost basis in the Growth Equity Fund
6. What is the average current price of securities in each sector?
7. For each portfolio, show the total value of Technology sector holdings and what percentage it represents of the total portfolio value
8. Find portfolios that have holdings in more than 5 different sectors and show their diversification metrics
9. What are the sector exposures for the Tech Innovation Fund?
10. Calculate the sector exposure breakdown for international equity


In [ ]:
TRUTH_DATA["questions"]

print(f'Select the question Number')
for item in TRUTH_DATA["questions"]:
    print(f'{item["id"]}. {item["question"]}')
user_input = int(input('Select the question Number'))
question_for_the_id = None
data_for_corresponding_id = None

In [8]:
for item in TRUTH_DATA["questions"]:
    if item["id"] == user_input:
        print(f'{item["id"]}. {item["question"]}')
        question_for_the_id= item["question"]
        data_for_corresponding_id=item
        break

1. How many portfolios do we have in total?


In [ ]:
question_for_the_id

'How many portfolios do we have in total?'

In [9]:
import sys

sys.path.append('d:\\Pl_Capitals_Assignment\\portfolio-analytics-agent')

In [10]:
from src.agent.langgraph_workflow import agent

response = agent.invoke(
        {
            "messages": [
                ("user", question_for_the_id)
            ]
        }
    )


print("\nAnswer:\n")
print(response["messages"][-1].content)


2026-04-05 21:10:38,436 | INFO | src.agent.langgraph_workflow | Running LLM node
2026-04-05 21:10:38,888 | INFO | src.database.db_connection | Connecting to database at d:\Pl_Capitals_Assignment\portfolio-analytics-agent\data\db\portfolio_database.db
2026-04-05 21:10:38,948 | INFO | src.database.db_connection | Database connected
2026-04-05 21:10:38,949 | INFO | src.database.db_connection | Dialect: sqlite
2026-04-05 21:10:38,950 | INFO | src.database.db_connection | Available tables: ['benchmarks', 'historical_prices', 'holdings', 'portfolio_performance', 'portfolios', 'risk_metrics', 'sectors', 'securities', 'transactions']
2026-04-05 21:10:40,684 | INFO | src.tools.sql_query_tool | User Question: SELECT COUNT(*) FROM portfolios;
2026-04-05 21:10:40,685 | INFO | src.tools.sql_query_tool | Generated SQL: SELECT COUNT(*) FROM portfolios;
2026-04-05 21:10:40,687 | INFO | src.tools.sql_query_tool | Query Result: [(13,)]
2026-04-05 21:10:40,690 | INFO | src.agent.langgraph_workflow | Runn


Answer:

There are 13 portfolios in total.


In [ ]:
import json

tool_msg = response["messages"][-2]

try:
    payload = json.loads(tool_msg.content)  
    sql_query = payload["sql_query"]
    result = payload["result"]
    
except(TypeError):
    sql_query, result = None, None

print("SQL:", sql_query)
print("Result:", result)

truth_sql_query = data_for_corresponding_id['ground_truth']['sql_query']

SQL: SELECT COUNT(portfolio_id) FROM portfolios;
Result: [[13]]


In [12]:
data_for_corresponding_id

{'id': 1,
 'type': 'text2sql',
 'difficulty': 'easy',
 'question': 'How many portfolios do we have in total?',
 'ground_truth': {'sql_query': 'SELECT COUNT(*) FROM portfolios;',
  'expected_result_type': 'single_value',
  'explanation': 'Simple COUNT query on portfolios table'}}

In [14]:
truth_sql_query

'SELECT COUNT(*) FROM portfolios;'

In [15]:
from src.database.db_connection import get_db_connection

In [20]:
curr = get_db_connection().cursor()

curr.execute(f'''{truth_sql_query}''')
row = curr.fetchall()

2026-04-05 21:15:08,957 | INFO | src.database.db_connection | Connected to database: D:\Pl_Capitals_Assignment\portfolio-analytics-agent\data\db\portfolio_database.db


In [21]:
row

[(13,)]